In [1]:
# --- Import libraries ---
import os
import requests
from openai import OpenAI
import json
import pandas as pd  # Add pandas for CSV saving
from dotenv import load_dotenv

In [2]:
# Specify the path to your .env file
load_dotenv("project_folder/LLM/API keys/API_keys.env")

# Now you can access your keys
deepseek_key = os.getenv("DEEPSEEK_API_KEY")

In [6]:
# --- DeepSeek API endpoint and model ---
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"
DEEPSEEK_MODEL = "deepseek-chat"

from prompt_deepseek import prompt as base_prompt

# --- DeepSeek query function ---
def deepseek_query(prompt, max_tokens=500):
    headers = {
        "Authorization": f"Bearer {deepseek_key}", # Use variable instead of os.environ
        "Content-Type": "application/json"
    }
    data = {
        "model": DEEPSEEK_MODEL,
        "messages": [
            {"role": "user", "content": str(prompt)}
        ],
        "max_tokens": max_tokens
    }
    response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data)
    print("Status code:", response.status_code)
    print("Response text:", response.text)
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]

In [ ]:
# --- Process JSON and save results as CSV ---
def process_json_with_deepseek(json_path, output_path="deepseek_json_outputs.csv"):
    """Process JSON file with preprocessed PDF chunks using DeepSeek."""
    try:
        # Check if file exists
        if not os.path.exists(json_path):
            print(f"❌ Error: File {json_path} not found")
            return
        
        with open(json_path) as f:
            chunks = json.load(f)
        
        print(f"✓ Loaded JSON with {len(chunks)} chunks from {json_path}")
        
        results = []
        for chunk in chunks:
            doc_id = chunk.get('doc_id', '')
            chunk_num = chunk.get('chunk', 0)
            content = chunk.get('text', '')
            
            # Skip empty content
            if not content.strip():
                continue
                
            print(f"Processing chunk {chunk_num} of document {doc_id}...")
            formatted_prompt = base_prompt.format(DOCUMENTATION=content)
            
            try:
                deepseek_output = deepseek_query(formatted_prompt, max_tokens=2000)
                
                # Try to parse JSON output for success tracking
                parsed_successfully = False
                try:
                    if "```json" in deepseek_output:
                        json_str = deepseek_output.split("```json")[1].split("```")[0].strip()
                        json.loads(json_str)
                        parsed_successfully = True
                    else:
                        json.loads(deepseek_output)
                        parsed_successfully = True
                except (json.JSONDecodeError, IndexError):
                    parsed_successfully = False
                
                results.append({
                    "document_id": doc_id,
                    "chunk_number": chunk_num,
                    "parsed_successfully": parsed_successfully,
                    "raw_output": deepseek_output.strip()
                })
                
            except Exception as e:
                results.append({
                    "document_id": doc_id,
                    "chunk_number": chunk_num,
                    "parsed_successfully": False,
                    "raw_output": f"Error: {e}"
                })
        
        # Save results to CSV
        df = pd.DataFrame(results)
        df.to_csv(output_path, index=False)
        print(f"✅ Processing complete. Results saved to {output_path}")
        
    except Exception as e:
        print(f"❌ Error processing JSON file: {e}")

# Check if the JSON file exists before processing
json_path = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs\27.json"

if os.path.exists(json_path):
    print(f"✓ Found file: {json_path}")
    process_json_with_deepseek(json_path, "doc_27_deepseek_results.csv")
else:
    print(f"❌ File not found: {json_path}")
    print("Available files in Json_docs directory:")
    json_dir = r"D:\OneDrive Files\OneDrive - University of Maryland\PhD research\Python\LLM\Json_docs"
    if os.path.exists(json_dir):
        files = [f for f in os.listdir(json_dir) if f.endswith('.json')]
        for f in sorted(files):
            print(f"   {f}")
    else:
        print(f"❌ Directory not found: {json_dir}")

Status code: 200
Response text: {"id":"f49b29e0-c3bd-4b26-8e4f-e2354e37cab3","object":"chat.completion","created":1746462969,"model":"deepseek-chat","choices":[{"index":0,"message":{"role":"assistant","content":"```json\n{\n  \"document_metadata\": {\n    \"title\": \"Revolution Wind, LLC Outer Continental Shelf Preconstruction Air Permit No. OCS-R1-05\",\n    \"document_number\": \"OCS-R1-05\",\n    \"Type of wind farm\": \"offshore\"\n  },\n  \"regulatory_constraints\": [\n    {\n      \"type\": \"Jurisdictional\",\n      \"requirement\": \"Compliance with Section 328 of the Clean Air Act and 40 C.F.R. Part 55\",\n      \"scope\": \"Federal waters on the Outer Continental Shelf\",\n      \"numerical_value\": null,\n      \"unit\": null,\n      \"source\": \"Clean Air Act, 40 C.F.R. Part 55\",\n      \"related_domains\": \"Air quality, regulatory compliance\"\n    },\n    {\n      \"type\": \"Technical\",\n      \"requirement\": \"Authorization to construct up to 100 wind turbine gene

In [ ]:
print("DEEPSEEK_API_KEY from os.environ:", os.environ.get("DEEPSEEK_API_KEY"))

DEEPSEEK_API_KEY from os.environ: sk-c3973e8bf7954f0899ebd0330da648fd
